# Doodle-to-Real Image Translation & Colorization using Pix2Pix

**Objective:** Design and implement a paired image-to-image translation system using a
**Conditional GAN (Pix2Pix)** that can:

1. Convert **Sketch / Edge images → Realistic images**
2. Convert **Grayscale images → Colored images**

The model learns a deterministic mapping between paired input-output images, generating
visually realistic outputs while preserving structural information.

| Component | Architecture | Details |
|-----------|-------------|---------|
| Generator | **U-Net** | Encoder-Decoder with skip connections |
| Discriminator | **PatchGAN** | 16×16 patch-level real/fake classification |
| GAN Loss | BCE | Adversarial signal |
| Reconstruction Loss | L1 | Pixel-level fidelity (λ = 100) |

**Datasets:**
- CUHK Face Sketch Database (CUFS)
- Anime Sketch Colorization Pair

**Platform:** Kaggle GPU T4 × 2

In [ ]:
# ============================================================
# 1. Environment Setup
# ============================================================
!pip install -q gradio scikit-image

import os, sys, random, time, glob, warnings
warnings.filterwarnings("ignore")

import torch
import torch.nn as nn
import torch.optim as optim
import torch.utils.data as data
import torchvision.transforms as T
import torchvision.utils as vutils
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
from skimage.metrics import structural_similarity as ssim
from skimage.metrics import peak_signal_noise_ratio as psnr

import matplotlib
matplotlib.rcParams['figure.dpi'] = 120
%matplotlib inline

# Reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

print(f"PyTorch  : {torch.__version__}")
print(f"CUDA     : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        props = torch.cuda.get_device_properties(i)
        print(f"  GPU {i}: {props.name}  ({props.total_mem / 1024**3:.1f} GB)")

In [ ]:
# ============================================================
# 2. Configuration & Hyperparameters
# ============================================================

# ── Dataset Selection ──────────────────────────────────────────────
# Set DATASET to "anime" or "cuhk" to choose which task to run.
# anime → Sketch → Color   (paired side-by-side images)
# cuhk  → Sketch → Photo   (separate folder pairs)
DATASET = "anime"

# Kaggle dataset paths (update to match your attached datasets)
ANIME_PATH = "/kaggle/input/anime-sketch-colorization-pair"
CUHK_PATH  = "/kaggle/input/cuhk-face-sketch-database-cufs"

# ── Architecture ───────────────────────────────────────────────────
IMAGE_SIZE   = 256       # Spatial resolution
NC_IN        = 3         # Input channels  (sketch/grayscale — will be 3-ch)
NC_OUT       = 3         # Output channels (colour photo)

# ── Training ───────────────────────────────────────────────────────
BATCH_SIZE   = 16        # Reduce to 8 if OOM on T4
NUM_EPOCHS   = 50        # Increase for better results
LR           = 0.0002
BETA1        = 0.5
BETA2        = 0.999
LAMBDA_L1    = 100       # Weight of L1 reconstruction loss

# ── Mixed Precision ───────────────────────────────────────────────
USE_AMP      = True

# ── Checkpointing ─────────────────────────────────────────────────
SAVE_EVERY   = 5
OUTPUT_DIR   = "/kaggle/working/"
os.makedirs(OUTPUT_DIR, exist_ok=True)

device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print(f"Device        : {device}")
print(f"GPUs          : {torch.cuda.device_count()}")
print(f"Dataset       : {DATASET}")
print(f"Image size    : {IMAGE_SIZE}x{IMAGE_SIZE}")
print(f"Batch size    : {BATCH_SIZE}")
print(f"Lambda L1     : {LAMBDA_L1}")

# 3. Data Preparation

**Paired datasets** are essential for Pix2Pix — every input sketch must have a corresponding
ground-truth real image.

We support two dataset formats:
1. **Anime Sketch Colorization** — Each image file is a side-by-side pair (left = color, right = sketch).
   The loader splits each image in half.
2. **CUHK Face Sketch** — Separate folders for sketches and photos. We match them by filename.

In [ ]:
# ============================================================
# 3. Data Preparation
# ============================================================

class PairedDataset(data.Dataset):
    """
    Flexible paired image dataset.
    
    Supports two modes:
    - 'side_by_side': Single image with input|target concatenated horizontally.
                      Left half = target (color), Right half = input (sketch).
    - 'separate_folders': Two directories with matching filenames.
    """

    def __init__(self, mode, size=256,
                 side_by_side_dir=None,
                 input_dir=None, target_dir=None):
        self.mode = mode
        self.size = size

        self.transform_input = T.Compose([
            T.Resize((size, size)),
            T.ToTensor(),
            T.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5)),
        ])
        self.transform_target = T.Compose([
            T.Resize((size, size)),
            T.ToTensor(),
            T.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5)),
        ])

        EXTS = {'.png', '.jpg', '.jpeg', '.bmp', '.webp'}

        if mode == 'side_by_side':
            assert side_by_side_dir is not None
            self.paths = sorted([
                os.path.join(dp, f)
                for dp, _, fns in os.walk(side_by_side_dir)
                for f in fns if os.path.splitext(f)[1].lower() in EXTS
            ])
        elif mode == 'separate_folders':
            assert input_dir is not None and target_dir is not None
            inp_files  = {os.path.splitext(f)[0]: os.path.join(input_dir, f)
                          for f in os.listdir(input_dir)
                          if os.path.splitext(f)[1].lower() in EXTS}
            tgt_files  = {os.path.splitext(f)[0]: os.path.join(target_dir, f)
                          for f in os.listdir(target_dir)
                          if os.path.splitext(f)[1].lower() in EXTS}
            common = sorted(set(inp_files.keys()) & set(tgt_files.keys()))
            self.pairs = [(inp_files[k], tgt_files[k]) for k in common]
        else:
            raise ValueError(f"Unknown mode: {mode}")

        length = len(self.paths) if mode == 'side_by_side' else len(self.pairs)
        print(f"[{mode}] Loaded {length} paired samples")

    def __len__(self):
        if self.mode == 'side_by_side':
            return len(self.paths)
        return len(self.pairs)

    def __getitem__(self, idx):
        if self.mode == 'side_by_side':
            img = Image.open(self.paths[idx]).convert('RGB')
            w, h = img.size
            # Left half = target (colour), Right half = input (sketch)
            target_img = img.crop((0, 0, w // 2, h))
            input_img  = img.crop((w // 2, 0, w, h))
        else:
            input_img  = Image.open(self.pairs[idx][0]).convert('RGB')
            target_img = Image.open(self.pairs[idx][1]).convert('RGB')

        inp = self.transform_input(input_img)
        tgt = self.transform_target(target_img)
        return inp, tgt

In [ ]:
# ── Build dataset based on configuration ──
if DATASET == "anime":
    # The anime-sketch-colorization-pair dataset has a 'data' subfolder
    # with train and val directories containing side-by-side paired images.
    train_dir = os.path.join(ANIME_PATH, "data", "train")
    val_dir   = os.path.join(ANIME_PATH, "data", "val")
    
    if not os.path.isdir(train_dir):
        # Fallback: try root directly
        train_dir = ANIME_PATH
        val_dir   = ANIME_PATH
        print(f"Warning: 'data/train' not found. Using root: {ANIME_PATH}")

    train_dataset = PairedDataset('side_by_side', size=IMAGE_SIZE,
                                  side_by_side_dir=train_dir)
    val_dataset   = PairedDataset('side_by_side', size=IMAGE_SIZE,
                                  side_by_side_dir=val_dir)

elif DATASET == "cuhk":
    # CUHK has separate photo and sketch folders.
    # Common structure: .../photos/ and .../sketches/
    photo_dir  = None
    sketch_dir = None
    for root, dirs, _ in os.walk(CUHK_PATH):
        for d in dirs:
            dl = d.lower()
            if 'photo' in dl:
                photo_dir = os.path.join(root, d)
            elif 'sketch' in dl:
                sketch_dir = os.path.join(root, d)
    assert photo_dir and sketch_dir, (
        f"Could not find photo/sketch subdirectories in {CUHK_PATH}. "
        f"Contents: {os.listdir(CUHK_PATH)}"
    )
    print(f"Photos  : {photo_dir}")
    print(f"Sketches: {sketch_dir}")

    full_ds = PairedDataset('separate_folders', size=IMAGE_SIZE,
                            input_dir=sketch_dir, target_dir=photo_dir)
    # 90/10 train-val split
    n_val = max(1, len(full_ds) // 10)
    n_train = len(full_ds) - n_val
    train_dataset, val_dataset = data.random_split(
        full_ds, [n_train, n_val],
        generator=torch.Generator().manual_seed(SEED)
    )
else:
    raise ValueError(f"Unknown DATASET: {DATASET}")

train_loader = data.DataLoader(train_dataset, batch_size=BATCH_SIZE,
                               shuffle=True, num_workers=2,
                               pin_memory=True, drop_last=True)
val_loader   = data.DataLoader(val_dataset, batch_size=BATCH_SIZE,
                               shuffle=False, num_workers=2,
                               pin_memory=True, drop_last=False)

print(f"\nTrain batches : {len(train_loader)}")
print(f"Val batches   : {len(val_loader)}")

In [ ]:
# ── Visualise some training pairs ──
inp_batch, tgt_batch = next(iter(train_loader))
n_show = min(6, inp_batch.size(0))

fig, axes = plt.subplots(2, n_show, figsize=(3 * n_show, 6))
for j in range(n_show):
    # Input (sketch)
    img_in = inp_batch[j].permute(1, 2, 0).numpy() * 0.5 + 0.5
    axes[0, j].imshow(img_in.clip(0, 1))
    axes[0, j].set_title("Input (Sketch)", fontsize=9)
    axes[0, j].axis("off")
    # Target (real)
    img_tgt = tgt_batch[j].permute(1, 2, 0).numpy() * 0.5 + 0.5
    axes[1, j].imshow(img_tgt.clip(0, 1))
    axes[1, j].set_title("Target (Real)", fontsize=9)
    axes[1, j].axis("off")

plt.suptitle("Sample Training Pairs", fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

# 4. Model Architecture

## 4.1 Generator — U-Net with Skip Connections

The U-Net has a symmetric encoder-decoder structure. Each encoder layer is connected to the
corresponding decoder layer via **skip connections** (concatenation), which preserves spatial
details that would otherwise be lost during downsampling.

```
Encoder                    Decoder
────────                   ────────
E1: 3→64   (256→128)  ──skip──►  D7: 512→64   (128→256)  → Tanh
E2: 64→128  (128→64)  ──skip──►  D6: 1024→128 (64→128)
E3: 128→256 (64→32)   ──skip──►  D5: 512→256  (32→64)
E4: 256→512 (32→16)   ──skip──►  D4: 1024→512 (16→32)
E5: 512→512 (16→8)    ──skip──►  D3: 1024→512 (8→16)
E6: 512→512 (8→4)     ──skip──►  D2: 1024→512 (4→8)
E7: 512→512 (4→2)     ──skip──►  D1: 1024→512 (2→4)
E8: 512→512 (2→1)     ─bottleneck─
```

## 4.2 Discriminator — PatchGAN

Instead of classifying the entire image as real/fake, PatchGAN classifies **overlapping
patches** of the image. This forces the discriminator to focus on **local texture realism**
rather than global structure (which the L1 loss already handles).

The discriminator receives a **concatenation** of the input sketch and the real/fake output
(6 channels total).

In [ ]:
# ============================================================
# 4. Model Definitions
# ============================================================

def weights_init(m):
    cn = m.__class__.__name__
    if 'Conv' in cn:
        nn.init.normal_(m.weight.data, 0.0, 0.02)
    elif 'BatchNorm' in cn:
        if m.weight is not None:
            nn.init.normal_(m.weight.data, 1.0, 0.02)
        if m.bias is not None:
            nn.init.constant_(m.bias.data, 0)


# ────────────────────── U-Net Generator ──────────────────────

class UNetDown(nn.Module):
    """Encoder block: Conv → [BN] → LeakyReLU"""
    def __init__(self, in_ch, out_ch, normalize=True):
        super().__init__()
        layers = [nn.Conv2d(in_ch, out_ch, 4, 2, 1, bias=False)]
        if normalize:
            layers.append(nn.BatchNorm2d(out_ch))
        layers.append(nn.LeakyReLU(0.2, inplace=True))
        self.model = nn.Sequential(*layers)

    def forward(self, x):
        return self.model(x)


class UNetUp(nn.Module):
    """Decoder block: ConvTranspose → BN → [Dropout] → ReLU"""
    def __init__(self, in_ch, out_ch, dropout=False):
        super().__init__()
        layers = [
            nn.ConvTranspose2d(in_ch, out_ch, 4, 2, 1, bias=False),
            nn.BatchNorm2d(out_ch),
        ]
        if dropout:
            layers.append(nn.Dropout(0.5))
        layers.append(nn.ReLU(inplace=True))
        self.model = nn.Sequential(*layers)

    def forward(self, x, skip):
        x = self.model(x)
        return torch.cat([x, skip], dim=1)   # skip connection


class UNetGenerator(nn.Module):
    """
    U-Net Generator for 256×256 images.
    8 encoder layers, 7 decoder layers + final output layer.
    """
    def __init__(self, in_ch=NC_IN, out_ch=NC_OUT):
        super().__init__()
        # ── Encoder ──
        self.e1 = UNetDown(in_ch, 64,  normalize=False)   # 256→128
        self.e2 = UNetDown(64,   128)                      # 128→64
        self.e3 = UNetDown(128,  256)                      # 64→32
        self.e4 = UNetDown(256,  512)                      # 32→16
        self.e5 = UNetDown(512,  512)                      # 16→8
        self.e6 = UNetDown(512,  512)                      # 8→4
        self.e7 = UNetDown(512,  512)                      # 4→2
        self.e8 = UNetDown(512,  512, normalize=False)     # 2→1 (bottleneck)

        # ── Decoder ──
        self.d1 = UNetUp(512,  512, dropout=True)   # 1→2,  cat with e7 → 1024
        self.d2 = UNetUp(1024, 512, dropout=True)   # 2→4,  cat with e6 → 1024
        self.d3 = UNetUp(1024, 512, dropout=True)   # 4→8,  cat with e5 → 1024
        self.d4 = UNetUp(1024, 512)                 # 8→16, cat with e4 → 1024
        self.d5 = UNetUp(1024, 256)                 # 16→32,cat with e3 → 512
        self.d6 = UNetUp(512,  128)                 # 32→64,cat with e2 → 256
        self.d7 = UNetUp(256,  64)                  # 64→128,cat with e1 → 128

        self.final = nn.Sequential(
            nn.ConvTranspose2d(128, out_ch, 4, 2, 1),  # 128→256
            nn.Tanh(),
        )

    def forward(self, x):
        # Encoder
        e1 = self.e1(x)
        e2 = self.e2(e1)
        e3 = self.e3(e2)
        e4 = self.e4(e3)
        e5 = self.e5(e4)
        e6 = self.e6(e5)
        e7 = self.e7(e6)
        e8 = self.e8(e7)

        # Decoder with skip connections
        d1 = self.d1(e8, e7)
        d2 = self.d2(d1, e6)
        d3 = self.d3(d2, e5)
        d4 = self.d4(d3, e4)
        d5 = self.d5(d4, e3)
        d6 = self.d6(d5, e2)
        d7 = self.d7(d6, e1)

        return self.final(d7)


# ────────────────────── PatchGAN Discriminator ──────────────────────

class PatchGAN(nn.Module):
    """
    PatchGAN discriminator.
    Input: concatenation of sketch (NC_IN) + image (NC_OUT) = 6 channels.
    Output: 16×16 matrix of real/fake probabilities.
    """
    def __init__(self, in_ch=NC_IN + NC_OUT):
        super().__init__()
        self.model = nn.Sequential(
            # Layer 1: no BatchNorm
            nn.Conv2d(in_ch, 64, 4, 2, 1),            # 256→128
            nn.LeakyReLU(0.2, inplace=True),

            # Layer 2
            nn.Conv2d(64, 128, 4, 2, 1, bias=False),  # 128→64
            nn.BatchNorm2d(128),
            nn.LeakyReLU(0.2, inplace=True),

            # Layer 3
            nn.Conv2d(128, 256, 4, 2, 1, bias=False), # 64→32
            nn.BatchNorm2d(256),
            nn.LeakyReLU(0.2, inplace=True),

            # Layer 4
            nn.Conv2d(256, 512, 4, 2, 1, bias=False), # 32→16
            nn.BatchNorm2d(512),
            nn.LeakyReLU(0.2, inplace=True),

            # Final classification layer
            nn.Conv2d(512, 1, 4, 1, 1),                # 16→16  (same-ish padding)
            nn.Sigmoid(),
        )

    def forward(self, input_img, target_img):
        x = torch.cat([input_img, target_img], dim=1)
        return self.model(x)

In [ ]:
# ── Instantiate models ──
netG = UNetGenerator().to(device)
netD = PatchGAN().to(device)

if torch.cuda.device_count() > 1:
    print(f"Using DataParallel across {torch.cuda.device_count()} GPUs")
    netG = nn.DataParallel(netG)
    netD = nn.DataParallel(netD)

netG.apply(weights_init)
netD.apply(weights_init)

# ── Print summaries ──
print("=" * 60)
print("U-NET GENERATOR")
print("=" * 60)
print(netG)
print(f"\nTotal parameters: {sum(p.numel() for p in netG.parameters()):,}")

print("\n" + "=" * 60)
print("PATCHGAN DISCRIMINATOR")
print("=" * 60)
print(netD)
print(f"\nTotal parameters: {sum(p.numel() for p in netD.parameters()):,}")

# Verify output shapes
with torch.no_grad():
    dummy_in = torch.randn(1, NC_IN, IMAGE_SIZE, IMAGE_SIZE, device=device)
    dummy_out = netG(dummy_in)
    print(f"\nGenerator   : {list(dummy_in.shape)} → {list(dummy_out.shape)}")
    dummy_d = netD(dummy_in, dummy_out)
    print(f"Discriminator: concat {list(dummy_in.shape)}+{list(dummy_out.shape)} → {list(dummy_d.shape)}")

# 5. Training

## Loss Functions
The Pix2Pix total loss for the **Generator** is:

$$\mathcal{L}_G = \underbrace{\text{BCE}(D(x, G(x)),\ 1)}_{\text{Adversarial}} + \lambda \cdot \underbrace{\| y - G(x) \|_1}_{\text{Reconstruction (L1)}}$$

- **Adversarial loss** pushes G to fool D (textures, local realism).
- **L1 loss** preserves the overall structure and colour fidelity (λ = 100).

The **Discriminator** is trained with standard BCE on real pairs vs fake pairs.

In [ ]:
# ============================================================
# 5. Training Setup
# ============================================================

criterion_GAN = nn.BCELoss()
criterion_L1  = nn.L1Loss()

optG = optim.Adam(netG.parameters(), lr=LR, betas=(BETA1, BETA2))
optD = optim.Adam(netD.parameters(), lr=LR, betas=(BETA1, BETA2))

scaler_g = torch.cuda.amp.GradScaler(enabled=USE_AMP)
scaler_d = torch.cuda.amp.GradScaler(enabled=USE_AMP)

def gpu_mem():
    if not torch.cuda.is_available():
        return
    for i in range(torch.cuda.device_count()):
        a = torch.cuda.memory_allocated(i) / 1024**2
        r = torch.cuda.memory_reserved(i) / 1024**2
        print(f"  GPU {i}: {a:.0f} MB alloc / {r:.0f} MB reserved")

print("Training configuration ready.")
gpu_mem()

In [ ]:
# ============================================================
# Training Loop
# ============================================================

hist = {
    "G_total": [], "G_gan": [], "G_l1": [], "D": [],
    "G_epoch": [], "D_epoch": [],
}
val_images = []   # Store (input, generated, target) tuples per epoch

print(f"Starting Pix2Pix training for {NUM_EPOCHS} epochs...")
t0 = time.time()

for epoch in range(NUM_EPOCHS):
    netG.train()
    netD.train()
    ep_g, ep_d, n = 0.0, 0.0, 0

    for i, (inp, tgt) in enumerate(train_loader):
        inp = inp.to(device)
        tgt = tgt.to(device)
        bs  = inp.size(0)

        # ═════════════════════════════════════════════════
        # (1) Update Discriminator
        # ═════════════════════════════════════════════════
        netD.zero_grad()

        with torch.cuda.amp.autocast(enabled=USE_AMP):
            # Real pair
            pred_real  = netD(inp, tgt)
            label_real = torch.ones_like(pred_real, device=device)
            loss_d_real = criterion_GAN(pred_real, label_real)

            # Fake pair
            fake        = netG(inp)
            pred_fake   = netD(inp, fake.detach())
            label_fake  = torch.zeros_like(pred_fake, device=device)
            loss_d_fake = criterion_GAN(pred_fake, label_fake)

            loss_d = (loss_d_real + loss_d_fake) * 0.5

        scaler_d.scale(loss_d).backward()
        scaler_d.step(optD)
        scaler_d.update()

        # ═════════════════════════════════════════════════
        # (2) Update Generator
        # ═════════════════════════════════════════════════
        netG.zero_grad()

        with torch.cuda.amp.autocast(enabled=USE_AMP):
            fake       = netG(inp)
            pred_fake  = netD(inp, fake)
            label_real = torch.ones_like(pred_fake, device=device)

            loss_g_gan = criterion_GAN(pred_fake, label_real)
            loss_g_l1  = criterion_L1(fake, tgt) * LAMBDA_L1
            loss_g     = loss_g_gan + loss_g_l1

        scaler_g.scale(loss_g).backward()
        scaler_g.step(optG)
        scaler_g.update()

        # ── Track ──
        hist["G_total"].append(loss_g.item())
        hist["G_gan"].append(loss_g_gan.item())
        hist["G_l1"].append(loss_g_l1.item())
        hist["D"].append(loss_d.item())
        ep_g += loss_g.item()
        ep_d += loss_d.item()
        n += 1

        if i % 50 == 0:
            print(f"  [{epoch+1}/{NUM_EPOCHS}][{i}/{len(train_loader)}]  "
                  f"D: {loss_d.item():.4f}  G: {loss_g.item():.4f}  "
                  f"(GAN={loss_g_gan.item():.4f}, L1={loss_g_l1.item():.4f})")

    hist["G_epoch"].append(ep_g / n)
    hist["D_epoch"].append(ep_d / n)

    # ── Validation snapshot ──
    netG.eval()
    with torch.no_grad():
        vi, vt = next(iter(val_loader))
        vi, vt = vi.to(device), vt.to(device)
        vf = netG(vi)
        val_images.append((
            vi[:8].cpu(), vf[:8].cpu(), vt[:8].cpu()
        ))

    # ── Checkpoint ──
    if (epoch + 1) % SAVE_EVERY == 0:
        torch.save(netG.state_dict(), os.path.join(OUTPUT_DIR, f"pix2pix_G_ep{epoch+1}.pth"))
        torch.save(netD.state_dict(), os.path.join(OUTPUT_DIR, f"pix2pix_D_ep{epoch+1}.pth"))
        print(f"  ✓ Checkpoint saved (epoch {epoch+1})")
        gpu_mem()

elapsed = time.time() - t0
torch.save(netG.state_dict(), os.path.join(OUTPUT_DIR, "pix2pix_G_final.pth"))
torch.save(netD.state_dict(), os.path.join(OUTPUT_DIR, "pix2pix_D_final.pth"))
print(f"\n✔ Training complete — {elapsed/60:.1f} min")

# 6. Visualisation & Evaluation

## 6.1 Training Loss Curves

In [ ]:
# ============================================================
# 6.1 Loss Curves
# ============================================================
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Per-iteration
ax = axes[0]
ax.set_title("Per-Iteration Losses", fontsize=12)
ax.plot(hist["G_total"], alpha=0.3, color="tab:blue", label="G Total")
ax.plot(hist["D"],       alpha=0.3, color="tab:orange", label="D")
ax.set_xlabel("Iteration"); ax.set_ylabel("Loss"); ax.legend()

# Epoch-averaged
ax = axes[1]
ax.set_title("Epoch-Averaged Losses", fontsize=12)
ax.plot(hist["G_epoch"], marker="o", color="tab:blue", label="Generator")
ax.plot(hist["D_epoch"], marker="s", color="tab:orange", label="Discriminator")
ax.set_xlabel("Epoch"); ax.set_ylabel("Loss"); ax.legend()

# G components
ax = axes[2]
ax.set_title("Generator Loss Components", fontsize=12)
# Compute epoch averages for GAN and L1 components
iters_per_epoch = len(train_loader)
g_gan_epochs = [np.mean(hist["G_gan"][e*iters_per_epoch:(e+1)*iters_per_epoch])
                for e in range(NUM_EPOCHS) if (e+1)*iters_per_epoch <= len(hist["G_gan"])]
g_l1_epochs  = [np.mean(hist["G_l1"][e*iters_per_epoch:(e+1)*iters_per_epoch])
                for e in range(NUM_EPOCHS) if (e+1)*iters_per_epoch <= len(hist["G_l1"])]
ax.plot(g_gan_epochs, marker="^", color="tab:green", label="Adversarial (GAN)")
ax.plot(g_l1_epochs,  marker="v", color="tab:red",   label=f"L1 × {LAMBDA_L1}")
ax.set_xlabel("Epoch"); ax.set_ylabel("Loss"); ax.legend()

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "loss_curves.png"), dpi=150)
plt.show()

## 6.2 Input → Generated → Ground Truth Comparison

In [ ]:
# ============================================================
# 6.2 Visual Comparison (last epoch)
# ============================================================

def show_triplets(inputs, fakes, targets, n=8, title=""):
    """Show Input | Generated | Ground Truth side by side."""
    n = min(n, inputs.size(0))
    fig, axes = plt.subplots(3, n, figsize=(2.5 * n, 7.5))
    
    row_labels = ["Input (Sketch)", "Generated", "Ground Truth"]
    for row, (imgs, label) in enumerate(zip([inputs, fakes, targets], row_labels)):
        for col in range(n):
            img = imgs[col].permute(1, 2, 0).numpy() * 0.5 + 0.5
            axes[row, col].imshow(img.clip(0, 1))
            axes[row, col].axis("off")
            if col == 0:
                axes[row, col].set_ylabel(label, fontsize=11, rotation=0,
                                           labelpad=80, va='center')
    if title:
        plt.suptitle(title, fontsize=14, y=1.02)
    plt.tight_layout()
    return fig

# Show final epoch results
if val_images:
    vi, vf, vt = val_images[-1]
    fig = show_triplets(vi, vf, vt, n=8, title="Final Epoch — Pix2Pix Results")
    fig.savefig(os.path.join(OUTPUT_DIR, "comparison_final.png"), dpi=150, bbox_inches="tight")
    plt.show()

## 6.3 Training Progression

In [ ]:
# ============================================================
# 6.3 Training Progression (snapshots across epochs)
# ============================================================
if val_images:
    n_snaps = len(val_images)
    step = max(1, n_snaps // 5)
    indices = list(range(0, n_snaps, step))
    if (n_snaps - 1) not in indices:
        indices.append(n_snaps - 1)

    fig, axes = plt.subplots(len(indices), 3, figsize=(8, 2.8 * len(indices)))
    if len(indices) == 1:
        axes = axes[np.newaxis, :]

    for row, idx in enumerate(indices):
        vi, vf, vt = val_images[idx]
        for col, (img, label) in enumerate(zip(
            [vi[0], vf[0], vt[0]],
            ["Input", "Generated", "Ground Truth"]
        )):
            arr = img.permute(1, 2, 0).numpy() * 0.5 + 0.5
            axes[row, col].imshow(arr.clip(0, 1))
            axes[row, col].axis("off")
            if col == 0:
                axes[row, col].set_ylabel(f"Epoch {idx+1}", fontsize=10)
            if row == 0:
                axes[row, col].set_title(label, fontsize=11)

    plt.suptitle("Training Progression", fontsize=14, y=1.01)
    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR, "training_progression.png"), dpi=150, bbox_inches="tight")
    plt.show()

## 6.4 Quantitative Evaluation — SSIM & PSNR

- **SSIM (Structural Similarity Index):** Measures perceived quality; ranges from -1 to 1 (higher = better).
- **PSNR (Peak Signal-to-Noise Ratio):** Measures pixel-level fidelity in dB (higher = better; >20 dB is decent for generative models).

In [ ]:
# ============================================================
# 6.4 Quantitative Evaluation — SSIM & PSNR
# ============================================================

def compute_metrics(generator, loader, device, max_batches=20):
    """Compute SSIM and PSNR over the validation set."""
    generator.eval()
    ssim_scores = []
    psnr_scores = []

    with torch.no_grad():
        for batch_idx, (inp, tgt) in enumerate(loader):
            if batch_idx >= max_batches:
                break
            inp = inp.to(device)
            tgt = tgt.to(device)
            fake = generator(inp)

            # Convert to numpy [0, 1]
            fake_np = (fake.cpu().numpy() * 0.5 + 0.5).clip(0, 1)
            tgt_np  = (tgt.cpu().numpy()  * 0.5 + 0.5).clip(0, 1)

            for j in range(fake_np.shape[0]):
                img_fake = np.transpose(fake_np[j], (1, 2, 0))  # H, W, C
                img_real = np.transpose(tgt_np[j],  (1, 2, 0))

                s = ssim(img_real, img_fake, multichannel=True,
                         channel_axis=2, data_range=1.0)
                p = psnr(img_real, img_fake, data_range=1.0)
                ssim_scores.append(s)
                psnr_scores.append(p)

    generator.train()
    return np.array(ssim_scores), np.array(psnr_scores)

ssim_scores, psnr_scores = compute_metrics(netG, val_loader, device)

print(f"{'Metric':<12} {'Mean':>10} {'Std':>10} {'Min':>10} {'Max':>10}")
print("-" * 54)
print(f"{'SSIM':<12} {ssim_scores.mean():>10.4f} {ssim_scores.std():>10.4f} "
      f"{ssim_scores.min():>10.4f} {ssim_scores.max():>10.4f}")
print(f"{'PSNR (dB)':<12} {psnr_scores.mean():>10.2f} {psnr_scores.std():>10.2f} "
      f"{psnr_scores.min():>10.2f} {psnr_scores.max():>10.2f}")

In [ ]:
# ── Distribution plots ──
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].hist(ssim_scores, bins=30, color="tab:blue", edgecolor="black", alpha=0.7)
axes[0].axvline(ssim_scores.mean(), color="red", ls="--", lw=2,
                label=f"Mean = {ssim_scores.mean():.4f}")
axes[0].set_title("SSIM Distribution", fontsize=13)
axes[0].set_xlabel("SSIM"); axes[0].set_ylabel("Count"); axes[0].legend()

axes[1].hist(psnr_scores, bins=30, color="tab:green", edgecolor="black", alpha=0.7)
axes[1].axvline(psnr_scores.mean(), color="red", ls="--", lw=2,
                label=f"Mean = {psnr_scores.mean():.2f} dB")
axes[1].set_title("PSNR Distribution", fontsize=13)
axes[1].set_xlabel("PSNR (dB)"); axes[1].set_ylabel("Count"); axes[1].legend()

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "ssim_psnr.png"), dpi=150)
plt.show()

# 7. Interactive App Deployment (Gradio)

Upload a sketch or grayscale image and watch the Pix2Pix model translate it into a realistic output in real-time.

In [ ]:
# ============================================================
# 7. Gradio App
# ============================================================
import gradio as gr

def pix2pix_inference(input_image):
    """
    Accepts a PIL image (sketch/grayscale), resizes to 256×256,
    runs through the trained generator, and returns the output.
    """
    if input_image is None:
        return None

    # Preprocess
    input_pil = input_image.convert("RGB")
    transform_in = T.Compose([
        T.Resize((IMAGE_SIZE, IMAGE_SIZE)),
        T.ToTensor(),
        T.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5)),
    ])
    inp_tensor = transform_in(input_pil).unsqueeze(0).to(device)

    # Generate
    netG.eval()
    with torch.no_grad():
        output = netG(inp_tensor)
    netG.train()

    # Post-process
    out_np = output.squeeze(0).cpu().permute(1, 2, 0).numpy() * 0.5 + 0.5
    out_np = (out_np.clip(0, 1) * 255).astype(np.uint8)
    return Image.fromarray(out_np)


def pix2pix_compare(input_image):
    """Generate and also return the input side-by-side."""
    result = pix2pix_inference(input_image)
    return input_image, result


# ── Build Gradio UI ──
with gr.Blocks(title="Pix2Pix: Sketch → Real", theme=gr.themes.Soft()) as demo:
    gr.Markdown(
        "## ✏️ Pix2Pix — Sketch-to-Real / Grayscale-to-Color\n"
        "Upload a sketch or grayscale image and watch the model generate a realistic version."
    )

    with gr.Tab("Generate"):
        with gr.Row():
            with gr.Column():
                img_input = gr.Image(label="Input (Sketch / Grayscale)", type="pil")
                gen_btn   = gr.Button("🎨 Generate", variant="primary")
            with gr.Column():
                img_output = gr.Image(label="Generated Output", type="pil")
        gen_btn.click(fn=pix2pix_inference, inputs=img_input, outputs=img_output)

    with gr.Tab("Validation Samples"):
        gr.Markdown("Random samples from the validation set.")
        val_btn = gr.Button("🔄 Show Validation Samples", variant="secondary")
        gallery = gr.Gallery(label="Input | Generated | Ground Truth", columns=3)

        def show_val_samples():
            netG.eval()
            images = []
            vi, vt = next(iter(val_loader))
            vi, vt = vi[:6].to(device), vt[:6].to(device)
            with torch.no_grad():
                vf = netG(vi)
            for j in range(vi.size(0)):
                for tensor in [vi[j], vf[j], vt[j]]:
                    arr = tensor.cpu().permute(1, 2, 0).numpy() * 0.5 + 0.5
                    arr = (arr.clip(0, 1) * 255).astype(np.uint8)
                    images.append(Image.fromarray(arr))
            netG.train()
            return images

        val_btn.click(fn=show_val_samples, outputs=gallery)

try:
    demo.launch(share=True, inline=False)
except Exception as e:
    print(f"Gradio launch failed: {e}")
    print("Fallback: demo.launch(inline=True)")

# 8. Summary & Conclusions

## How Pix2Pix Works

| Step | What Happens |
|------|-------------|
| 1 | A **sketch/edge/grayscale** image is fed into the U-Net Generator |
| 2 | The Generator produces a **realistic/coloured** output |
| 3 | The PatchGAN Discriminator sees the **(input, output)** pair and classifies each 16×16 patch as real or fake |
| 4 | The Generator is penalised by both the **adversarial loss** (fool the Discriminator) and the **L1 loss** (stay close to ground truth) |

## Key Design Decisions

| Choice | Rationale |
|--------|-----------|
| **U-Net skip connections** | Preserve fine spatial details (edges, textures) that would be lost in a plain encoder-decoder |
| **PatchGAN** | Focuses the discriminator on local texture quality; cheaper than a full-image discriminator |
| **L1 loss (λ=100)** | Prevents the generator from hallucinating unrealistic content; maintains structural fidelity |
| **Mixed Precision** | Fits the model within Kaggle T4 memory constraints |

## Quantitative Metrics

| Metric | What It Measures |
|--------|-----------------|
| **SSIM** | Structural similarity between generated and ground truth images |
| **PSNR** | Pixel-level fidelity (higher dB = closer to ground truth) |

## Files Saved
- `pix2pix_G_final.pth` / `pix2pix_D_final.pth` — Final model weights
- `loss_curves.png` — Training loss plots  
- `comparison_final.png` — Input vs Generated vs Ground Truth
- `ssim_psnr.png` — Metric distribution histograms
- `training_progression.png` — How outputs improve over epochs